# Week 3 — Statistics for ML: Measures of Spread

**Difficulty:** ⭐⭐⭐ (Intermediate)  
**Estimated Time:** 2.5 hours  
**Theme:** House Price Prediction  

---

> **What you will learn:**  
> Variance, standard deviation, skewness, and kurtosis — the four statistics that describe **how spread out and shaped** your data is. You will also learn why these statistics directly affect which ML models you can (and cannot) use on a feature.

## 1. Why Does Spread Matter for Machine Learning?

Two neighborhoods can have the **exact same mean price** but tell completely different stories:

| | Neighborhood A | Neighborhood B |
|---|---|---|
| Mean price | $300,000 | $300,000 |
| Prices | $295K, $298K, $302K, $305K | $100K, $200K, $350K, $550K |
| Standard deviation | ~$4,000 | ~$175,000 |

**For a buyer:** Neighborhood A is predictable (stable investment). Neighborhood B is risky (high variance).

**For an ML model:** These two columns have the same mean but different scales. Without understanding spread:
- **Feature scaling** (StandardScaler) subtracts the mean and divides by std dev — wrong std dev = wrong scaling
- **Linear regression** assumes normally distributed residuals — heavy-tailed errors violate this
- **Gradient descent** converges much slower on high-variance, unevenly-scaled features
- **Skewed features** can cause linear models to underfit and tree models to make inefficient splits

## 2. Real-World Analogy: The Temperature vs Earthquake Scale

**Standard deviation** is like measuring daily temperature fluctuations in two cities:

- **London:** Mean temp = 12°C, daily variation = ±3°C → small std dev, predictable
- **Moscow:** Mean temp = 5°C, daily variation = ±18°C → large std dev, wildly unpredictable

The mean alone does not help you choose what to wear tomorrow. The **spread** does.

**Skewness** is like asking: "Is the earthquake distribution symmetric?" Most days have zero tremors (the peak is at 0). But rarely a magnitude-9 quake happens (a very long right tail). That asymmetry is positive skew.

House prices behave exactly like earthquakes — most are clustered in a moderate range, but a small number of luxury properties create a long right tail.

## 3. Setup — Creating Two Neighborhoods and a Skewed Dataset

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')   # Suppress minor scipy warnings during the lesson

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)

# ── DATASET 1: Two Neighborhoods with the SAME mean but DIFFERENT spread ──────

# Neighborhood A: Stable, uniform pricing (low standard deviation)
# Think: a planned suburban development where all houses are similar
n_houses = 200
neighborhood_A = np.random.normal(loc=300_000, scale=15_000, size=n_houses)

# Neighborhood B: Volatile, mixed pricing (high standard deviation)
# Think: a gentrifying urban area with old apartments next to new luxury flats
neighborhood_B = np.random.normal(loc=300_000, scale=100_000, size=n_houses)

# Clip both to avoid negative prices
neighborhood_A = np.clip(neighborhood_A, 50_000, None)
neighborhood_B = np.clip(neighborhood_B, 50_000, None)

# ── DATASET 2: Skewed distribution using lognormal ────────────────────────────
# np.random.lognormal creates a right-skewed distribution —
# this is the natural shape of real-world house prices because
# prices cannot go below $0 but can go very high.
# Parameters chosen so the resulting prices centre around ~$300K.
log_prices = np.random.lognormal(mean=12.6, sigma=0.5, size=500)

# ── Wrap everything in a DataFrame for analysis ───────────────────────────────
df_neigh = pd.DataFrame({
    'Neighborhood_A': neighborhood_A,
    'Neighborhood_B': neighborhood_B
})
df_skewed = pd.DataFrame({'price': log_prices})

print("Datasets created:")
print(f"  Neighborhood A: n={n_houses}, mean=${np.mean(neighborhood_A):,.0f}, std=${np.std(neighborhood_A):,.0f}")
print(f"  Neighborhood B: n={n_houses}, mean=${np.mean(neighborhood_B):,.0f}, std=${np.std(neighborhood_B):,.0f}")
print(f"  Skewed prices : n=500, mean=${log_prices.mean():,.0f}, median=${np.median(log_prices):,.0f}")

## 4. Variance — The Average Squared Deviation

### Plain English First

Variance answers: **"On average, how far is each house price from the mean price?"**

Why do we **square** the deviations instead of just averaging them?

If we did not square, positive deviations (houses above the mean) and negative deviations (houses below the mean) would cancel each other out, always giving zero. Squaring eliminates negatives and also penalises large deviations more than small ones.

### Formula

**Population variance** (when you have ALL the data):

$$\sigma^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i - \bar{x})^2$$

**Sample variance** (when you have a sample from a larger population, default in pandas/numpy ddof=1):

$$s^2 = \frac{1}{n-1} \sum_{i=1}^{n} (x_i - \bar{x})^2$$

The $n-1$ (Bessel's correction) adjusts for the fact that a sample tends to underestimate the true variance.

### Units Problem

If prices are in dollars, variance is in **dollars squared** — an unnatural unit. A variance of $1,000,000,000 does not intuitively mean anything. That is why we take the square root to get standard deviation.

In [ ]:
# ── Variance: Manual Step-by-Step Calculation ─────────────────────────────────
# We will compute variance for Neighborhood A step by step.

prices = neighborhood_A    # Alias for readability
n = len(prices)

# Step 1: Compute the mean
mean_price = np.mean(prices)
print(f"Step 1 — Mean price : ${mean_price:,.0f}")

# Step 2: Compute each deviation from the mean  (x_i - mean)
deviations = prices - mean_price
print(f"Step 2 — First 5 deviations from mean: {deviations[:5].round(0)}")
print(f"         Sum of raw deviations: {deviations.sum():.4f}  (approximately 0 — they cancel out!)") 

# Step 3: Square each deviation  (x_i - mean)^2
# Squaring makes all values positive and amplifies large deviations
squared_devs = deviations ** 2
print(f"\nStep 3 — First 5 squared deviations: {squared_devs[:5].round(0)}")

# Step 4: Average the squared deviations (population variance, divide by n)
variance_manual = np.sum(squared_devs) / n    # Population variance
print(f"\nStep 4 — Population variance : ${variance_manual:,.0f} dollars^2")
print(f"         (This is in dollars-squared — hard to interpret directly)")

# ── Cross-check with NumPy (ddof=0 = population variance) ────────────────────
variance_numpy = np.var(prices, ddof=0)
print(f"\nNumPy variance (ddof=0) : ${variance_numpy:,.0f}")

# ── Sample variance (ddof=1, used when we have a sample, not the full population) ──
variance_sample = np.var(prices, ddof=1)
print(f"NumPy variance (ddof=1) : ${variance_sample:,.0f}  (sample variance, slightly larger)")

# Compare neighborhoods
print("\n" + "="*50)
print("VARIANCE COMPARISON: A vs B")
print(f"  Neighborhood A variance: ${np.var(neighborhood_A):>18,.0f} $/^2")
print(f"  Neighborhood B variance: ${np.var(neighborhood_B):>18,.0f} $/^2")
print(f"  B is {np.var(neighborhood_B)/np.var(neighborhood_A):.0f}x more variable than A — same mean, wildly different spread")

## 5. Standard Deviation — Back in the Original Units

### Plain English First

Standard deviation is simply the **square root of variance**. By taking the square root, we convert dollars-squared back to dollars — a unit we can actually reason about.

**Intuition:** Standard deviation is the "typical distance" any single house price sits from the mean.

- If std = $15,000: most houses are within $15K of the mean — a tight, predictable market
- If std = $100,000: most houses are within $100K of the mean — a wild, unpredictable market

### Formula

$$\sigma = \sqrt{\sigma^2} = \sqrt{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^2}$$

### The 68-95-99.7 Rule (Empirical Rule)

For a **normally distributed** feature:
- ~68% of values fall within **1 std dev** of the mean (mean ± σ)
- ~95% of values fall within **2 std devs** of the mean (mean ± 2σ)
- ~99.7% of values fall within **3 std devs** of the mean (mean ± 3σ)

This rule is why ML practitioners use "3-sigma" as a common outlier threshold.

In [ ]:
# ── Standard Deviation: Manual and NumPy ──────────────────────────────────────

std_A_manual = np.sqrt(variance_manual)   # Square root of the variance we computed above
std_A_numpy  = np.std(neighborhood_A, ddof=0)   # NumPy cross-check
std_B        = np.std(neighborhood_B, ddof=0)

mean_A = np.mean(neighborhood_A)
mean_B = np.mean(neighborhood_B)

print("STANDARD DEVIATION COMPARISON")
print(f"  Neighborhood A std : ${std_A_numpy:,.0f}")
print(f"  Neighborhood B std : ${std_B:,.0f}")
print(f"  Manual vs NumPy agreement: {round(std_A_manual) == round(std_A_numpy)}")

# ── Verify the 68-95-99.7 Rule on Neighborhood A ─────────────────────────────
prices_A = neighborhood_A
print("\n68-95-99.7 RULE VERIFICATION (Neighborhood A, nearly normal):")

for sigma_mult, expected_pct in [(1, 68), (2, 95), (3, 99.7)]:
    lower = mean_A - sigma_mult * std_A_numpy    # Lower bound
    upper = mean_A + sigma_mult * std_A_numpy    # Upper bound
    # Count what fraction of houses fall within this band
    within = np.mean((prices_A >= lower) & (prices_A <= upper)) * 100
    print(f"  Within {sigma_mult}σ (${lower:,.0f} to ${upper:,.0f}): "
          f"{within:.1f}%  (theory: ~{expected_pct}%)")

# ── Show std dev as 'typical distance from the mean' ─────────────────────────
print("\nSTD DEV AS 'TYPICAL DISTANCE FROM THE MEAN':")
avg_distance_A = np.mean(np.abs(prices_A - mean_A))   # Mean absolute deviation
print(f"  Neighborhood A — mean abs deviation : ${avg_distance_A:,.0f}")
print(f"  Neighborhood A — std deviation      : ${std_A_numpy:,.0f}")
print(f"  These are similar — std dev ≈ typical distance from the mean.")
print(f"  (They differ because std dev squares deviations before averaging.)")

## 6. Visualizing the Two Neighborhoods: Same Mean, Different Spread

In [ ]:
# ── Side-by-side histograms of the two neighborhoods ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)
fig.suptitle('Same Mean, Different Standard Deviation — Why Spread Matters',
             fontsize=14, fontweight='bold')

neighborhoods = [
    (neighborhood_A, 'steelblue',  'Neighborhood A (Stable)',
     f'Mean=${mean_A:,.0f}\nStd=${std_A_numpy:,.0f}'),
    (neighborhood_B, 'darkorange', 'Neighborhood B (Volatile)',
     f'Mean=${mean_B:,.0f}\nStd=${std_B:,.0f}'),
]

for ax, (prices, color, title, label) in zip(axes, neighborhoods):
    mean_val = np.mean(prices)
    std_val  = np.std(prices)
    
    # Draw the histogram
    ax.hist(prices, bins=30, color=color, edgecolor='white', alpha=0.8)
    
    # Mark the mean with a vertical dashed line
    ax.axvline(mean_val, color='black', linewidth=2, linestyle='--',
               label=f'Mean = ${mean_val:,.0f}')
    
    # Shade the ±1 std dev band
    ax.axvspan(mean_val - std_val, mean_val + std_val,
               alpha=0.2, color='yellow', label=f'±1σ = ${std_val:,.0f}')
    
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('House Price ($)', fontsize=10)
    ax.set_ylabel('Count', fontsize=10)
    ax.legend(fontsize=9)
    ax.xaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K')
    )
    ax.tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

print("KEY OBSERVATION:")
print(f"  Both neighborhoods have mean ≈ $300K.")
print(f"  Neighborhood A: houses cluster tightly — the ±1σ band is narrow (${std_A_numpy:,.0f})")
print(f"  Neighborhood B: houses spread widely  — the ±1σ band is wide  (${std_B:,.0f})")
print(f"\n  An ML model trained on A alone would give confident, accurate predictions.")
print(f"  The same model applied to B would have much higher prediction error.")

## 7. Skewness — Measuring Asymmetry

### Plain English First

A symmetric distribution looks like a perfect bell curve: the left side is a mirror image of the right side. Most ML model assumptions (e.g., linear regression residuals) rely on this symmetry.

**Skewness** measures how lopsided the distribution is:

- **Skewness = 0:** Perfectly symmetric (normal distribution)
- **Skewness > 0 (positive / right skew):** Long tail on the right. A few very high values pull the mean above the median. **House prices are typically right-skewed** because luxury homes create a long right tail, but prices cannot go below $0.
- **Skewness < 0 (negative / left skew):** Long tail on the left. Think: exam scores where most students score high but a few score very low.

### Rule of Thumb

| Skewness value | Interpretation |
|---|---|
| -0.5 to +0.5 | Approximately symmetric — safe for most models |
| ±0.5 to ±1.0 | Moderately skewed — consider transformation |
| > +1 or < -1 | Highly skewed — log transform likely needed |

### Formula (Fisher's moment coefficient of skewness)

$$\text{Skewness} = \frac{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^3}{\sigma^3}$$

It is a normalised third moment. Cubing preserves the sign of the deviation (positive or negative), so asymmetry in either direction is captured.

In [ ]:
# ── Skewness: Manual Calculation and Scipy ────────────────────────────────────

def compute_skewness_manual(x):
    """Compute Fisher's skewness coefficient from scratch."""
    n    = len(x)
    mean = np.mean(x)
    std  = np.std(x, ddof=0)      # Population std (ddof=0) for this formula
    
    # Compute the normalised third central moment
    # (x_i - mean)^3 preserves sign: large positive outliers → large positive contribution
    third_moment = np.mean((x - mean) ** 3)
    skewness     = third_moment / (std ** 3)   # Divide by std^3 to normalise
    return skewness

# Compute for three distributions
skew_A = compute_skewness_manual(neighborhood_A)
skew_B = compute_skewness_manual(neighborhood_B)
skew_log = compute_skewness_manual(log_prices)

print("SKEWNESS RESULTS (Manual)")
print(f"  Neighborhood A (normal, low std)    : skewness = {skew_A:+.3f}")
print(f"  Neighborhood B (normal, high std)   : skewness = {skew_B:+.3f}")
print(f"  Log-normal prices (right-skewed)    : skewness = {skew_log:+.3f}")

# Cross-check with scipy.stats.skew
print("\nSKEWNESS RESULTS (scipy.stats.skew)")
# scipy uses a bias-corrected version (slightly different for small n)
print(f"  Neighborhood A  : {stats.skew(neighborhood_A):+.3f}")
print(f"  Neighborhood B  : {stats.skew(neighborhood_B):+.3f}")
print(f"  Log-normal prices: {stats.skew(log_prices):+.3f}")

# Interpret the results
print("\nINTERPRETATION:")
print(f"  A and B are nearly normal (skewness ≈ 0) — symmetric distributions.")
print(f"  Log-normal skewness = {stats.skew(log_prices):.2f} → highly right-skewed.")
print(f"  This means: a small number of very expensive homes pull the distribution right.")
print(f"  Real house price datasets typically have skewness in the range +0.8 to +2.5.")

## 8. Kurtosis — Measuring Tail Weight

### Plain English First

While skewness measures left-right asymmetry, **kurtosis** measures how heavy the tails are — how likely you are to see extreme values.

Imagine two bell curves with the same mean and std dev:
- **Thin tails (low kurtosis / platykurtic):** Most values are close to the mean, extreme values are very rare. Like the height distribution of professional basketball players — everyone is tall, few extremes.
- **Fat tails (high kurtosis / leptokurtic):** Extreme values occur more often than expected. Like daily stock returns — most days are quiet, but occasionally a massive crash or rally happens.

**Excess kurtosis** subtracts 3 (the kurtosis of a normal distribution) so that:
- Excess kurtosis = 0: normal distribution (baseline)
- Excess kurtosis > 0: heavy tails (leptokurtic) — more extreme outliers than expected
- Excess kurtosis < 0: light tails (platykurtic) — fewer extreme outliers than expected

### Why Kurtosis Matters for ML

Many ML algorithms assume normally distributed features or residuals. **Heavy tails violate this assumption.** A linear regression with leptokurtic residuals will have:
- Unreliable confidence intervals
- Predictions that are occasionally wildly wrong for extreme properties
- Poor performance on anomaly detection

### Formula (Excess Kurtosis)

$$\text{Excess Kurtosis} = \frac{\frac{1}{n}\sum_{i=1}^{n}(x_i - \bar{x})^4}{\sigma^4} - 3$$

In [ ]:
# ── Kurtosis: Manual Calculation and Scipy ────────────────────────────────────

def compute_kurtosis_manual(x):
    """Compute excess kurtosis from scratch."""
    n    = len(x)
    mean = np.mean(x)
    std  = np.std(x, ddof=0)        # Population std
    
    # Fourth central moment: (x_i - mean)^4  (always positive, so large outliers dominate)
    fourth_moment = np.mean((x - mean) ** 4)
    
    # Divide by std^4 to normalise, then subtract 3 to get EXCESS kurtosis
    # (the normal distribution has kurtosis=3, so subtracting 3 gives us
    #  a baseline of 0 for comparison)
    excess_kurtosis = fourth_moment / (std ** 4) - 3
    return excess_kurtosis

# Compute for all three datasets
kurt_A   = compute_kurtosis_manual(neighborhood_A)
kurt_B   = compute_kurtosis_manual(neighborhood_B)
kurt_log = compute_kurtosis_manual(log_prices)

print("EXCESS KURTOSIS RESULTS (Manual)")
print(f"  Neighborhood A (normal)      : {kurt_A:+.3f}")
print(f"  Neighborhood B (normal)      : {kurt_B:+.3f}")
print(f"  Log-normal prices (skewed)   : {kurt_log:+.3f}")

# scipy.stats.kurtosis returns EXCESS kurtosis by default (fisher=True by default)
print("\nEXCESS KURTOSIS RESULTS (scipy.stats.kurtosis)")
print(f"  Neighborhood A  : {stats.kurtosis(neighborhood_A, fisher=True):+.3f}")
print(f"  Neighborhood B  : {stats.kurtosis(neighborhood_B, fisher=True):+.3f}")
print(f"  Log-normal prices: {stats.kurtosis(log_prices, fisher=True):+.3f}")

print("\nINTERPRETATION:")
print(f"  A and B are near-normal (excess kurtosis ≈ 0) — tails as expected.")
print(f"  Log-normal prices: kurtosis = {stats.kurtosis(log_prices):.2f}")
print(f"  Positive excess kurtosis → HEAVIER tails than normal → more extreme prices.")
print(f"  This means the luxury tail is fatter than a Gaussian — extreme prices occur")
print(f"  more frequently than a normal distribution would predict.")

## 9. Complete Visualization — Distribution Shape Analysis

In [ ]:
# ── Four-panel figure: Compare normal vs skewed price distributions ───────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Distribution Shape Analysis — Skewness and Kurtosis in House Prices',
             fontsize=14, fontweight='bold')

# ── Panel 1: Neighborhood A — Normal, Low Spread ─────────────────────────────
ax = axes[0, 0]
ax.hist(neighborhood_A, bins=30, color='steelblue', edgecolor='white', alpha=0.8,
        density=True)   # density=True: y-axis shows probability density, not count
ax.axvline(np.mean(neighborhood_A),   color='crimson',    lw=2, ls='--', label='Mean')
ax.axvline(np.median(neighborhood_A), color='darkorange', lw=2, ls='-',  label='Median')
ax.set_title(f'Neighborhood A (Stable)\nSkew={stats.skew(neighborhood_A):+.2f}, '
             f'KurtEx={stats.kurtosis(neighborhood_A):+.2f}', fontsize=10)
ax.set_xlabel('Price ($)'); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e3:.0f}K'))

# ── Panel 2: Log-normal — Right-Skewed Distribution ──────────────────────────
ax = axes[0, 1]
ax.hist(log_prices, bins=50, color='darkorange', edgecolor='white', alpha=0.8,
        density=True)
ax.axvline(np.mean(log_prices),   color='crimson',    lw=2, ls='--', label=f'Mean=${np.mean(log_prices)/1e3:.0f}K')
ax.axvline(np.median(log_prices), color='navy',       lw=2, ls='-',  label=f'Median=${np.median(log_prices)/1e3:.0f}K')
ax.set_title(f'Log-Normal Prices (Right-Skewed)\nSkew={stats.skew(log_prices):+.2f}, '
             f'KurtEx={stats.kurtosis(log_prices):+.2f}', fontsize=10)
ax.set_xlabel('Price ($)'); ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M' if x>=1e6 else f'${x/1e3:.0f}K'))

# ── Panel 3: Q-Q Plot for Neighborhood A (should be near-normal) ─────────────
ax = axes[1, 0]
# Q-Q plot: each point is a quantile of our data vs the expected normal quantile.
# If data is normal, points fall on the diagonal reference line.
(osm, osr), (slope, intercept, r) = stats.probplot(neighborhood_A, dist='norm')
ax.scatter(osm, osr, color='steelblue', s=15, alpha=0.6, label='Data quantiles')
line_x = np.array([min(osm), max(osm)])
ax.plot(line_x, slope * line_x + intercept, 'r--', lw=2, label='Normal reference line')
ax.set_title('Q-Q Plot: Neighborhood A\n(Near-normal → points hug the line)', fontsize=10)
ax.set_xlabel('Theoretical Quantiles'); ax.set_ylabel('Sample Quantiles')
ax.legend(fontsize=9)

# ── Panel 4: Q-Q Plot for Log-normal (should deviate from line) ───────────────
ax = axes[1, 1]
(osm2, osr2), (slope2, intercept2, r2) = stats.probplot(log_prices, dist='norm')
ax.scatter(osm2, osr2, color='darkorange', s=15, alpha=0.6, label='Data quantiles')
line_x2 = np.array([min(osm2), max(osm2)])
ax.plot(line_x2, slope2 * line_x2 + intercept2, 'r--', lw=2, label='Normal reference line')
ax.set_title('Q-Q Plot: Log-Normal Prices\n(Right tail curves up → NOT normal)', fontsize=10)
ax.set_xlabel('Theoretical Quantiles'); ax.set_ylabel('Sample Quantiles')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Q-Q PLOT GUIDE:")
print("  Points on the diagonal → data is normally distributed")
print("  Right tail curves UP   → right skew (heavy right tail)")
print("  Left tail curves DOWN  → left skew (heavy left tail)")
print("  S-shaped curve         → leptokurtic (heavy both tails)")

## 10. Log Transformation — Fixing Skewed House Prices

### Plain English First

Many ML models (linear regression, neural networks with certain loss functions) work best when features are **approximately normally distributed**.

Since house prices are right-skewed (a few mansions create a long right tail), we can apply a **log transformation** to compress the right tail and make the distribution more symmetric.

**Why does log work?** Taking $\log(x)$ compresses large values more than small ones:

| Original price | log(price) |
|---|---|
| $100,000 | 11.51 |
| $300,000 | 12.61 |
| $1,000,000 | 13.82 |
| $3,000,000 | 14.91 |

A $3M mansion is 30x a $100K house, but in log space it is only 1.3 units away instead of 2.9M units away. This "squeezes" the right tail.

### Important Caveats

- Log transform requires all values to be **positive** (prices satisfy this)
- After transforming, your model predicts **log(price)** — you must apply `exp()` to get back to dollars
- The log transformation is appropriate for **multiplicative processes** — price appreciation compounds multiplicatively, so log is the natural scale

In [ ]:
# ── Log Transformation: Before and After ──────────────────────────────────────

# Original right-skewed prices
original_prices = log_prices.copy()

# Apply natural logarithm (base e)
# np.log computes ln(x). We could also use np.log10 (base 10), but ln is standard in ML.
log_transformed = np.log(original_prices)

# Compute before/after statistics
print("BEFORE LOG TRANSFORMATION (original prices):")
print(f"  Mean     : ${np.mean(original_prices):,.0f}")
print(f"  Median   : ${np.median(original_prices):,.0f}")
print(f"  Std Dev  : ${np.std(original_prices):,.0f}")
print(f"  Skewness : {stats.skew(original_prices):+.3f}  (highly right-skewed)")
print(f"  Kurtosis : {stats.kurtosis(original_prices):+.3f}  (heavy right tail)")

print("\nAFTER LOG TRANSFORMATION (log prices):")
print(f"  Mean     : {np.mean(log_transformed):.3f}  (now in log-dollar units)")
print(f"  Median   : {np.median(log_transformed):.3f}")
print(f"  Std Dev  : {np.std(log_transformed):.3f}")
print(f"  Skewness : {stats.skew(log_transformed):+.3f}  (nearly symmetric now!)")
print(f"  Kurtosis : {stats.kurtosis(log_transformed):+.3f}")

# Demonstrate round-tripping: log → exp should give back original
reconstructed = np.exp(log_transformed)   # Reverse the log: e^(log(x)) = x
print(f"\nROUND-TRIP CHECK:")
print(f"  Original  first 3: {original_prices[:3].round(0)}")
print(f"  Recovered first 3: {reconstructed[:3].round(0)}")
print(f"  Max error: ${np.max(np.abs(original_prices - reconstructed)):.6f}  (essentially zero)")

In [ ]:
# ── Visualization: Log Transformation Effect ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Log Transformation: Reducing Skewness in House Prices',
             fontsize=14, fontweight='bold')

# ── Left: Original skewed prices ──────────────────────────────────────────────
ax1 = axes[0]
ax1.hist(original_prices, bins=50, color='tomato', edgecolor='white', alpha=0.8,
         density=True)
ax1.axvline(np.mean(original_prices),   color='black',  lw=2.0, ls='--',
            label=f'Mean   ${np.mean(original_prices)/1e3:.0f}K')
ax1.axvline(np.median(original_prices), color='navy',   lw=2.0, ls='-',
            label=f'Median ${np.median(original_prices)/1e3:.0f}K')
ax1.set_title(f'BEFORE: Original Prices\n'
              f'Skewness = {stats.skew(original_prices):+.2f} (highly skewed)', fontsize=11)
ax1.set_xlabel('House Price ($)', fontsize=10)
ax1.set_ylabel('Density', fontsize=10)
ax1.legend(fontsize=9)
ax1.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x,_: f'${x/1e6:.1f}M' if x>=1e6 else f'${x/1e3:.0f}K')
)
ax1.tick_params(axis='x', rotation=30)

# ── Right: Log-transformed prices ─────────────────────────────────────────────
ax2 = axes[1]
ax2.hist(log_transformed, bins=40, color='seagreen', edgecolor='white', alpha=0.8,
         density=True)

# Overlay a normal distribution curve on the transformed data
x_range = np.linspace(log_transformed.min(), log_transformed.max(), 200)
# stats.norm.pdf(x, loc=mean, scale=std) gives the theoretical normal density
normal_curve = stats.norm.pdf(x_range,
                               loc=np.mean(log_transformed),
                               scale=np.std(log_transformed))
ax2.plot(x_range, normal_curve, 'k-', lw=2, label='Normal curve (reference)')

ax2.axvline(np.mean(log_transformed),   color='black', lw=2.0, ls='--',
            label=f'Mean   {np.mean(log_transformed):.2f}')
ax2.axvline(np.median(log_transformed), color='navy',  lw=2.0, ls='-',
            label=f'Median {np.median(log_transformed):.2f}')
ax2.set_title(f'AFTER: Log(Price)\n'
              f'Skewness = {stats.skew(log_transformed):+.2f} (nearly symmetric!)', fontsize=11)
ax2.set_xlabel('log(House Price)', fontsize=10)
ax2.set_ylabel('Density', fontsize=10)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("After the log transform:")
print("  The distribution is much more bell-shaped.")
print("  Mean ≈ Median → the skewness is gone.")
print("  The normal reference curve fits well.")
print("  This transformed feature is now safe to use in linear models.")

## 11. Putting It All Together — Complete Shape Analysis

In [ ]:
# ── Full Statistical Profile: All Spread Measures Together ───────────────────

def full_profile(name, data):
    """Print a complete spread and shape profile for a dataset."""
    mean   = np.mean(data)
    std    = np.std(data, ddof=1)    # Sample std dev (ddof=1 is the pandas default)
    var    = np.var(data, ddof=1)
    skew   = stats.skew(data)
    kurt   = stats.kurtosis(data, fisher=True)   # Excess kurtosis

    print(f"\n{name}")
    print("-" * 45)
    print(f"  Mean             : ${mean:>12,.2f}")
    print(f"  Variance         : ${var:>12,.2f} (dollars^2)")
    print(f"  Std Dev          : ${std:>12,.2f}")
    print(f"  Skewness         : {skew:>+12.4f}", end="  ")
    if abs(skew) < 0.5:
        print("← symmetric")
    elif abs(skew) < 1.0:
        print(f"← moderately {'right' if skew>0 else 'left'}-skewed")
    else:
        print(f"← HIGHLY {'right' if skew>0 else 'left'}-skewed")
    print(f"  Excess Kurtosis  : {kurt:>+12.4f}", end="  ")
    if abs(kurt) < 0.5:
        print("← normal tails")
    elif kurt > 0:
        print("← HEAVY tails (leptokurtic)")
    else:
        print("← light tails (platykurtic)")

print("COMPLETE SHAPE ANALYSIS")
print("=" * 45)
full_profile("Neighborhood A (Stable, Low Variance)", neighborhood_A)
full_profile("Neighborhood B (Volatile, High Variance)", neighborhood_B)
full_profile("Log-Normal Prices (Right-Skewed, Raw)", log_prices)
full_profile("Log-Normal Prices (After Log Transform)", log_transformed)

print("\n" + "=" * 45)
print("ML IMPLICATIONS:")
print("  A: Ready for StandardScaler + linear models — low spread, symmetric")
print("  B: Same scaler works, but wider prediction intervals expected")
print("  Raw: Apply log transform BEFORE using in linear models")
print("  Log-transformed: Near-normal — linear regression will work well")

## 12. Quick Reference — Spread and Shape Statistics in ML Context

| Statistic | Formula | Tells You | ML Use Case |
|---|---|---|---|
| **Variance** | $\frac{1}{n}\sum(x_i-\bar{x})^2$ | Average squared spread | Feature selection: high variance features carry more information |
| **Std Dev** | $\sqrt{\text{Variance}}$ | Typical distance from mean | StandardScaler divides by std dev; bandwidth for KDE |
| **Skewness** | $\frac{E[(x-\mu)^3]}{\sigma^3}$ | Asymmetry of distribution | Flag features needing log/sqrt/Box-Cox transform |
| **Kurtosis** | $\frac{E[(x-\mu)^4]}{\sigma^4} - 3$ | Tail weight vs normal | Identify features that will violate normality assumptions |

**scikit-learn connection:**
- `StandardScaler`: removes mean, divides by std dev → uses variance/std dev directly
- `PowerTransformer(method='box-cox')`: reduces skewness automatically (generalised log transform)
- `PowerTransformer(method='yeo-johnson')`: same but works on negative values too
- `QuantileTransformer`: forces output to uniform or normal distribution regardless of kurtosis

## 13. Self-Check Questions

Answer these before reading the solutions below.

---

**Question 1:** Neighborhood A has std = $5,000; Neighborhood B has std = $100,000. Both have mean = $300,000. What does this tell you about each neighborhood, and which has more predictable prices?

**Question 2:** House prices are typically right-skewed. Why does this make intuitive sense given how real estate markets work?

**Question 3:** Before feeding house prices into a linear regression, should you apply a log transformation to right-skewed price data? What specific problem does the transformation fix?

**Question 4:** You have two features: `price` (std = $80,000) and `lot_size_sqft` (std = 500 sqft). If you feed both into gradient descent without scaling, what problem arises? How does std dev help you fix it?

**Question 5:** You compute skewness = +2.4 for a `price` column and skewness = -0.3 for a `room_count` column. Which one should you consider transforming, and which is acceptable as-is?

## 13. Answers

---

**Answer 1:**  
Neighborhood A (std = $5K) has a very tight, predictable market. Almost all houses (about 95% by the empirical rule) are priced between $290K and $310K. It is likely a uniform planned development. Neighborhood B (std = $100K) is volatile — a wide range of properties, from modest homes to expensive ones, all in the same area. A buyer could get a $100K property or a $500K one in the same neighborhood. **Neighborhood A has more predictable prices.** For an ML model, predictions for Neighborhood A will have much smaller error bars.

---

**Answer 2:**  
House prices cannot go below zero (a hard floor), but there is no ceiling — a mansion can cost $10M, $50M, or more. This asymmetry naturally creates a long right tail. Additionally, most people can only afford modest homes (clustering the bulk of prices in the moderate range), while a small wealthy population buys very expensive properties. Both the hard floor and the concentration of demand in the moderate range cause the right tail to be stretched much longer than the left tail, producing positive skewness.

---

**Answer 3:**  
Yes. A log transformation should be applied. Linear regression assumes that the **residuals** (prediction errors) are normally distributed. When the target variable (price) is right-skewed, the residuals will also tend to be skewed, violating this assumption. Consequences include: unreliable p-values and confidence intervals, the model penalising large over-predictions more than large under-predictions (because squared errors are not symmetric under skew), and poor model fit for typical prices when outliers dominate the loss. The log transform compresses the right tail and makes the distribution approximately normal, allowing linear regression to satisfy its assumptions.

---

**Answer 4:**  
Gradient descent updates weights proportionally to gradients. The `price` feature varies on a scale of tens of thousands; `lot_size_sqft` varies on a scale of hundreds. The gradient for `price` will be vastly larger, causing the optimizer to take huge steps in the price dimension and tiny steps in the lot_size dimension. This creates an elongated loss surface where gradient descent oscillates and converges very slowly. The fix: divide each feature by its standard deviation (StandardScaler). This puts both features on a comparable scale, making gradient descent take balanced steps in every direction and converging much faster.

---

**Answer 5:**  
**Transform `price` (skewness +2.4).** This exceeds the |1.0| threshold for high skewness. A log or Box-Cox transform is strongly recommended before using this feature in linear models or neural networks with MSE loss. **Keep `room_count` as-is (skewness -0.3).** This falls well within the ±0.5 "approximately symmetric" range — it is safe to use directly without transformation. Additionally, room count is a discrete integer (1, 2, 3, 4, 5 rooms) and transforming it would destroy its natural interpretation.

In [ ]:
# ── Final Recap: All Spread and Shape Statistics ──────────────────────────────
print("NOTEBOOK 2 — MEASURES OF SPREAD: FINAL SUMMARY")
print("=" * 55)

datasets = [
    ("Neigh. A (stable)",        neighborhood_A),
    ("Neigh. B (volatile)",      neighborhood_B),
    ("Log-normal (raw)",         log_prices),
    ("Log-normal (transformed)", log_transformed),
]

# Print a compact table
header = f"  {'Dataset':<28} {'Std Dev':>10}  {'Skew':>7}  {'Kurtosis':>9}"
print(header)
print("  " + "-" * 60)
for name, data in datasets:
    std_str = f"${np.std(data):,.0f}" if np.std(data) > 100 else f"{np.std(data):.3f}"
    print(f"  {name:<28} {std_str:>10}  "
          f"{stats.skew(data):>+7.3f}  "
          f"{stats.kurtosis(data):>+9.3f}")

print()
print("KEY TAKEAWAYS:")
print("  1. Variance and std dev measure how SPREAD OUT prices are.")
print("  2. Two neighborhoods can have identical means but very different spreads.")
print("  3. Skewness > +1 means log transform is likely needed.")
print("  4. High excess kurtosis means heavier tails than normal — more extreme values.")
print("  5. Log transform compresses the right tail and reduces skewness to near-zero.")
print("  6. StandardScaler uses mean and std dev — always check spread BEFORE scaling.")
print()
print("You are now ready for Notebook 03: Probability Distributions")